# Cardano Governance KG — SPARQL Queries
**KEN4256 · Tinaye Mawocha**

Loads `ontology.ttl` + `governance_kg.ttl` locally via rdflib and runs
all sanity-check and analytical queries.

> **Requirements:** `pip install rdflib pandas`

## Setup — load both TTL files

In [2]:
%pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 8.1 MB/s  0:00:01m0:00:010:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 12.2 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]2m1/2 [pandas]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import pandas as pd
from rdflib import ConjunctiveGraph

# ── Adjust this path if needed ────────────────────────────────────────
BASE = os.path.expanduser('~/Documents/UM/KGS/cardanokb')
ONTOLOGY_PATH = os.path.join(BASE, 'ontology.ttl')
KG_PATH       = os.path.join(BASE, 'governance_kg.ttl')
# ──────────────────────────────────────────────────────────────────────

g = ConjunctiveGraph()
print('Loading ontology …')
g.parse(ONTOLOGY_PATH, format='turtle')
print(f'  Ontology triples : {len(g):,}')

print('Loading knowledge graph …')
g.parse(KG_PATH, format='turtle')
print(f'  Total triples    : {len(g):,}')
print('Done.')

/var/folders/00/f2s2ptdj4917fxqcydf7b_ym0000gn/T/ipykernel_13825/3243392967.py:11: DeprecationWarning: ConjunctiveGraph is deprecated, use Dataset instead.
  g = ConjunctiveGraph()


Loading ontology …
  Ontology triples : 338
Loading knowledge graph …
  Total triples    : 1,590,478
Done.


## Helper — run a SPARQL query and return a DataFrame

In [4]:
def run(sparql: str, max_rows: int = 200) -> pd.DataFrame:
    """Execute a SPARQL SELECT and return results as a pandas DataFrame."""
    results = g.query(sparql)
    rows = []
    for row in results:
        rows.append({str(var): str(val) if val is not None else None
                     for var, val in zip(results.vars, row)})
    df = pd.DataFrame(rows)
    return df.head(max_rows)

In [15]:
import os

RESULTS_DIR = os.path.join(BASE, "query_results")
os.makedirs(RESULTS_DIR, exist_ok=True)

def run_and_save(name: str, sparql: str) -> pd.DataFrame:
    df = run(sparql)
    path = os.path.join(RESULTS_DIR, f"{name}.csv")
    df.to_csv(path, index=False)
    print(f"Saved {len(df)} rows → {path}")
    return df

---
## SC-1 · Entity counts
Confirms all 6 classes loaded correctly.

**Expected:** DRep≈1593, StakeAddress≈193072, Vote≈21923, Proposal=91, StakePool=100, Epoch=20

In [ ]:
SC1 = """
PREFIX cgov:   <https://w3id.org/cgov#>
PREFIX rdf:    <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs:   <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd:    <http://www.w3.org/2001/XMLSchema#>
PREFIX skos:   <http://www.w3.org/2004/02/skos/core#>
PREFIX owl:    <http://www.w3.org/2002/07/owl#>
PREFIX prov:   <http://www.w3.org/ns/prov#>
PREFIX schema: <https://schema.org/>
PREFIX dc:     <http://purl.org/dc/terms/>
SELECT ?class (COUNT(?entity) AS ?count)
WHERE {
  VALUES ?class {
    cgov:Epoch cgov:StakePool cgov:DRep
    cgov:StakeAddress cgov:Proposal cgov:Vote
  }
  ?entity rdf:type ?class .
}
GROUP BY ?class
ORDER BY DESC(?count)
"""

run(SC1)

TypeError: run_and_save() missing 1 required positional argument: 'sparql'

## SC-2 · DRep delegator + vote spot-check
Confirms that delegation and vote relations are wired to DRep nodes.

In [6]:
SC2 = """
PREFIX cgov:   <https://w3id.org/cgov#>
PREFIX rdf:    <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs:   <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd:    <http://www.w3.org/2001/XMLSchema#>
PREFIX skos:   <http://www.w3.org/2004/02/skos/core#>
PREFIX owl:    <http://www.w3.org/2002/07/owl#>
PREFIX prov:   <http://www.w3.org/ns/prov#>
PREFIX schema: <https://schema.org/>
PREFIX dc:     <http://purl.org/dc/terms/>
SELECT ?drep ?delegatorCount ?voteCount
WHERE {
  {
    SELECT ?drep (COUNT(DISTINCT ?addr) AS ?delegatorCount)
    WHERE {
      ?addr cgov:delegatesVotingPowerTo ?drep .
      ?drep rdf:type cgov:DRep .
    }
    GROUP BY ?drep
  }
  {
    SELECT ?drep (COUNT(DISTINCT ?vote) AS ?voteCount)
    WHERE {
      ?drep cgov:castVote ?vote .
    }
    GROUP BY ?drep
  }
}
ORDER BY DESC(?delegatorCount)
LIMIT 10
"""

run(SC2)

,drep,delegatorCount,voteCount
0,https://w3id.org/cgov#AlwaysAbstain,106400,NaN
1,https://w3id.org/cgov#drep_drep1ygr9tuapcanc3k...,18949,81
2,https://w3id.org/cgov#drep_drep1y29h2q6cst2pvk...,9208,83
3,https://w3id.org/cgov#AlwaysNoConfidence,6678,NaN
4,https://w3id.org/cgov#drep_drep1yt8p080ajks6zd...,6126,35
5,https://w3id.org/cgov#drep_drep1ytfwpmt2fvdnyv...,2581,57
6,https://w3id.org/cgov#drep_drep1y2m0g4r66pyaw3...,1744,87
7,https://w3id.org/cgov#drep_drep1y2rtf58m0lv9sp...,1700,64
8,https://w3id.org/cgov#drep_drep1yf2jzhuc4f7eu2...,1656,83
9,https://w3id.org/cgov#drep_drep1yftc8zs7gjcj4a...,1619,67


---
## Q1 · Top 10 DReps by delegated voting power
*Who holds the most voting power in Cardano governance?*

In [17]:
Q1 = """
PREFIX cgov:   <https://w3id.org/cgov#>
PREFIX rdf:    <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs:   <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd:    <http://www.w3.org/2001/XMLSchema#>
PREFIX skos:   <http://www.w3.org/2004/02/skos/core#>
PREFIX owl:    <http://www.w3.org/2002/07/owl#>
PREFIX prov:   <http://www.w3.org/ns/prov#>
PREFIX schema: <https://schema.org/>
PREFIX dc:     <http://purl.org/dc/terms/>
SELECT ?drep
       (SUM(?amountLovelace) AS ?totalLovelace)
       (ROUND(SUM(?amountLovelace) / 1000000) AS ?totalADA)
       (COUNT(?addr) AS ?numDelegators)
WHERE {
  ?addr rdf:type cgov:StakeAddress ;
        cgov:delegatesVotingPowerTo ?drep ;
        cgov:delegatedStakeLovelace ?amountLovelace .
  ?drep rdf:type cgov:DRep .
}
GROUP BY ?drep
ORDER BY DESC(?totalLovelace)
LIMIT 10
"""

run_and_save("Q1", Q1)

Saved 10 rows → /Users/tinayemawocha/Documents/UM/KGS/cardanokb/query_results/Q1.csv


,drep,totalLovelace,totalADA,numDelegators
0,https://w3id.org/cgov#AlwaysAbstain,5707883061194556,5707883061,106400
1,https://w3id.org/cgov#drep_drep1ygr9tuapcanc3k...,708055927053612,708055927,18949
2,https://w3id.org/cgov#drep_drep1y2200we9c904un...,456568809742071,456568810,1475
3,https://w3id.org/cgov#drep_drep1ytvlwvyjmzfyn5...,297548831433855,297548831,400
4,https://w3id.org/cgov#drep_drep1y2nvdau7g7xt6d...,292099619312895,292099619,90
5,https://w3id.org/cgov#drep_drep1y29h2q6cst2pvk...,248602562781261,248602563,9208
6,https://w3id.org/cgov#drep_drep1ytfwpmt2fvdnyv...,209815727793573,209815728,2581
7,https://w3id.org/cgov#AlwaysNoConfidence,193509003946402,193509004,6678
8,https://w3id.org/cgov#drep_drep1ydpfkyjxzeqval...,189935504651684,189935505,543
9,https://w3id.org/cgov#drep_drep1y2csyxt7u2hl46...,186436943451321,186436943,1368


## Q2 · Vote breakdown (Yes / No / Abstain) per proposal
*How did each governance proposal perform at the vote?*

In [18]:
Q2 = """
PREFIX cgov:   <https://w3id.org/cgov#>
PREFIX rdf:    <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs:   <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd:    <http://www.w3.org/2001/XMLSchema#>
PREFIX skos:   <http://www.w3.org/2004/02/skos/core#>
PREFIX owl:    <http://www.w3.org/2002/07/owl#>
PREFIX prov:   <http://www.w3.org/ns/prov#>
PREFIX schema: <https://schema.org/>
PREFIX dc:     <http://purl.org/dc/terms/>
SELECT ?proposal ?proposalType
       (SUM(IF(?opt = cgov:Yes,     1, 0)) AS ?yesCount)
       (SUM(IF(?opt = cgov:No,      1, 0)) AS ?noCount)
       (SUM(IF(?opt = cgov:Abstain, 1, 0)) AS ?abstainCount)
       (COUNT(?vote) AS ?totalVotes)
WHERE {
  ?vote rdf:type cgov:Vote ;
        cgov:voteOnProposal ?proposal ;
        cgov:hasVoteOption ?opt .
  OPTIONAL {
    ?proposal cgov:hasProposalType ?typeNode .
    ?typeNode skos:prefLabel ?proposalType .
  }
}
GROUP BY ?proposal ?proposalType
ORDER BY DESC(?totalVotes)
"""

run_and_save("Q2", Q2)

Saved 91 rows → /Users/tinayemawocha/Documents/UM/KGS/cardanokb/query_results/Q2.csv


,proposal,proposalType,yesCount,noCount,abstainCount,totalVotes
0,https://w3id.org/cgov#proposal_0b19476e40bbbb5...,Hard Fork Initiation,655,6,36,697
1,https://w3id.org/cgov#proposal_c21b00f90f18fce...,Parameter Change,540,15,83,638
2,https://w3id.org/cgov#proposal_47a0e7a4f9383b1...,New Committee,605,12,20,637
3,https://w3id.org/cgov#proposal_4dab331457b61b8...,New Committee,457,1,2,460
4,https://w3id.org/cgov#proposal_9b62b3c632f3290...,Info Action,248,184,18,450
...,...,...,...,...,...,...
86,https://w3id.org/cgov#proposal_2f2fc2539dde550...,Treasury Withdrawal,71,6,4,81
87,https://w3id.org/cgov#proposal_e9693ed6b6465b2...,Info Action,8,1,0,9
88,https://w3id.org/cgov#proposal_b2a591ac219ce6d...,Parameter Change,7,0,0,7
89,https://w3id.org/cgov#proposal_51f495aa23f4b3b...,Parameter Change,0,5,0,5


## Q3 · Distribution of proposals by type
*What kinds of on-chain decisions are being made?*

In [19]:
Q3 = """
PREFIX cgov:   <https://w3id.org/cgov#>
PREFIX rdf:    <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs:   <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd:    <http://www.w3.org/2001/XMLSchema#>
PREFIX skos:   <http://www.w3.org/2004/02/skos/core#>
PREFIX owl:    <http://www.w3.org/2002/07/owl#>
PREFIX prov:   <http://www.w3.org/ns/prov#>
PREFIX schema: <https://schema.org/>
PREFIX dc:     <http://purl.org/dc/terms/>
SELECT ?typeConcept ?typeLabel (COUNT(?proposal) AS ?count)
WHERE {
  ?proposal rdf:type cgov:Proposal ;
            cgov:hasProposalType ?typeConcept .
  ?typeConcept skos:prefLabel ?typeLabel .
}
GROUP BY ?typeConcept ?typeLabel
ORDER BY DESC(?count)
"""

run_and_save("Q3", Q3)

Saved 6 rows → /Users/tinayemawocha/Documents/UM/KGS/cardanokb/query_results/Q3.csv


,typeConcept,typeLabel,count
0,https://w3id.org/cgov#TreasuryWithdrawal,Treasury Withdrawal,48
1,https://w3id.org/cgov#InfoAction,Info Action,30
2,https://w3id.org/cgov#ParameterChange,Parameter Change,5
3,https://w3id.org/cgov#NewConstitution,New Constitution,4
4,https://w3id.org/cgov#NewCommittee,New Committee,3
5,https://w3id.org/cgov#HardForkInitiation,Hard Fork Initiation,1


## Q4 · Most active DReps by votes cast
*Which DReps are most engaged in on-chain governance?*

In [20]:
Q4 = """
PREFIX cgov:   <https://w3id.org/cgov#>
PREFIX rdf:    <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs:   <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd:    <http://www.w3.org/2001/XMLSchema#>
PREFIX skos:   <http://www.w3.org/2004/02/skos/core#>
PREFIX owl:    <http://www.w3.org/2002/07/owl#>
PREFIX prov:   <http://www.w3.org/ns/prov#>
PREFIX schema: <https://schema.org/>
PREFIX dc:     <http://purl.org/dc/terms/>
SELECT ?drep (COUNT(?vote) AS ?votesCast)
       (SAMPLE(?role) AS ?voterRole)
WHERE {
  ?drep cgov:castVote ?vote .
  ?vote cgov:voterRole ?role .
}
GROUP BY ?drep
ORDER BY DESC(?votesCast)
LIMIT 20
"""

run_and_save("Q4", Q4)

Saved 20 rows → /Users/tinayemawocha/Documents/UM/KGS/cardanokb/query_results/Q4.csv


,drep,votesCast,voterRole
0,https://w3id.org/cgov#drep_drep1y2jnqfamg4tq5a...,120,drep
1,https://w3id.org/cgov#drep_drep1y2j2q9pcl85596...,100,drep
2,https://w3id.org/cgov#drep_drep1y2zu74gt8sxtrx...,98,drep
3,https://w3id.org/cgov#drep_drep1ytxr0lhg5j72wy...,98,drep
4,https://w3id.org/cgov#drep_drep1yfp4qmp849250s...,97,drep
5,https://w3id.org/cgov#drep_drep1y2hlgh9600zjlt...,96,drep
6,https://w3id.org/cgov#drep_drep1y2u2848fer43gh...,95,drep
7,https://w3id.org/cgov#drep_drep1ytm2zedqf54prq...,94,drep
8,https://w3id.org/cgov#drep_drep1yt6lgdpqjey286...,94,drep
9,https://w3id.org/cgov#drep_drep1yfdfs28uwafjgm...,93,drep


## Q5 · Top 10 pools by live stake
*How decentralised is stake distribution among the top 100 pools?*

In [21]:
Q5 = """
PREFIX cgov:   <https://w3id.org/cgov#>
PREFIX rdf:    <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs:   <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd:    <http://www.w3.org/2001/XMLSchema#>
PREFIX skos:   <http://www.w3.org/2004/02/skos/core#>
PREFIX owl:    <http://www.w3.org/2002/07/owl#>
PREFIX prov:   <http://www.w3.org/ns/prov#>
PREFIX schema: <https://schema.org/>
PREFIX dc:     <http://purl.org/dc/terms/>
SELECT ?pool ?ticker
       (ROUND(?liveLovelace / 1000000) AS ?liveStakeADA)
       (ROUND(?pledgeLovelace / 1000000) AS ?declaredPledgeADA)
       (?marginFee AS ?margin)
WHERE {
  ?pool rdf:type cgov:StakePool ;
        cgov:liveStakeLovelace ?liveLovelace .
  OPTIONAL { ?pool cgov:poolTicker ?ticker . }
  OPTIONAL { ?pool cgov:declaredPledgeLovelace ?pledgeLovelace . }
  OPTIONAL { ?pool cgov:marginFee ?marginFee . }
}
ORDER BY DESC(?liveLovelace)
LIMIT 10
"""

run_and_save("Q5", Q5)

Saved 10 rows → /Users/tinayemawocha/Documents/UM/KGS/cardanokb/query_results/Q5.csv


,pool,ticker,liveStakeADA,declaredPledgeADA,margin
0,https://w3id.org/cgov#pool_pool1k3nkfa5ugavrpd...,None,95399628,100,0.05
1,https://w3id.org/cgov#pool_pool18rjrygm3knlt67...,None,76315167,76000000,1.0
2,https://w3id.org/cgov#pool_pool1n6erydn8x79sa3...,None,76301500,76000000,1.0
3,https://w3id.org/cgov#pool_pool1xmlq3sgssww6kw...,None,76301500,76000000,1.0
4,https://w3id.org/cgov#pool_pool1l0erdjregched3...,None,76301500,76000000,1.0
5,https://w3id.org/cgov#pool_pool1uy07qyf0gzzua2...,None,76300000,75500000,1.0
6,https://w3id.org/cgov#pool_pool1luhm55xxqp0cv7...,None,75500026,73000000,1.0
7,https://w3id.org/cgov#pool_pool1rkm8rplkafvllx...,None,75500010,75500000,1.0
8,https://w3id.org/cgov#pool_pool1r2fph82mjhgu95...,None,75500002,14750000,1.0
9,https://w3id.org/cgov#pool_pool1l92vxagssugcsu...,None,69466693,0,0.1


## Q6 · DRep participation gap — DReps with zero votes cast
*How many registered DReps have never voted on a proposal?*

In [22]:
Q6 = """
PREFIX cgov:   <https://w3id.org/cgov#>
PREFIX rdf:    <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs:   <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd:    <http://www.w3.org/2001/XMLSchema#>
PREFIX skos:   <http://www.w3.org/2004/02/skos/core#>
PREFIX owl:    <http://www.w3.org/2002/07/owl#>
PREFIX prov:   <http://www.w3.org/ns/prov#>
PREFIX schema: <https://schema.org/>
PREFIX dc:     <http://purl.org/dc/terms/>
SELECT (COUNT(?drep) AS ?drepCount) ?hasVoted
WHERE {
  ?drep rdf:type cgov:DRep .
  BIND(EXISTS { ?drep cgov:castVote ?v . } AS ?hasVoted)
}
GROUP BY ?hasVoted
"""

run_and_save("Q6", Q6)

Saved 2 rows → /Users/tinayemawocha/Documents/UM/KGS/cardanokb/query_results/Q6.csv


,drepCount,hasVoted
0,867,false
1,726,true


## Q7 · Dual-delegation overlap
*What fraction of stakers are actively participating in governance?*

Counts stake addresses by whether they delegate to a pool, a DRep, both, or neither.

In [23]:
Q7 = """
PREFIX cgov:   <https://w3id.org/cgov#>
PREFIX rdf:    <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs:   <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd:    <http://www.w3.org/2001/XMLSchema#>
PREFIX skos:   <http://www.w3.org/2004/02/skos/core#>
PREFIX owl:    <http://www.w3.org/2002/07/owl#>
PREFIX prov:   <http://www.w3.org/ns/prov#>
PREFIX schema: <https://schema.org/>
PREFIX dc:     <http://purl.org/dc/terms/>
SELECT
  (COUNT(?addr) AS ?totalStakeAddresses)
  (SUM(IF(?hasPool && ?hasDRep,  1, 0)) AS ?fullyParticipating)
  (SUM(IF(?hasPool && !?hasDRep, 1, 0)) AS ?stakingOnlyNoDRep)
  (SUM(IF(!?hasPool && ?hasDRep, 1, 0)) AS ?drepOnlyNoPool)
WHERE {
  ?addr rdf:type cgov:StakeAddress .
  BIND(EXISTS { ?addr cgov:delegatesStakeTo       ?pool . } AS ?hasPool)
  BIND(EXISTS { ?addr cgov:delegatesVotingPowerTo ?drep . } AS ?hasDRep)
}
"""

run_and_save("Q7", Q7)

Saved 1 rows → /Users/tinayemawocha/Documents/UM/KGS/cardanokb/query_results/Q7.csv


,totalStakeAddresses,fullyParticipating,stakingOnlyNoDRep,drepOnlyNoPool
0,193072,6364,3759,182949


---
## Bonus · Epoch timeline
Useful for the methodology / results section of the report.

In [24]:
Q8 = """
PREFIX cgov:   <https://w3id.org/cgov#>
PREFIX rdf:    <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs:   <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd:    <http://www.w3.org/2001/XMLSchema#>
PREFIX skos:   <http://www.w3.org/2004/02/skos/core#>
PREFIX owl:    <http://www.w3.org/2002/07/owl#>
PREFIX prov:   <http://www.w3.org/ns/prov#>
PREFIX schema: <https://schema.org/>
PREFIX dc:     <http://purl.org/dc/terms/>
SELECT ?epoch ?epochNumber ?startTime ?endTime
       (ROUND(?activeStake / 1000000000000) AS ?activeStakeTADA)
WHERE {
  ?epoch rdf:type cgov:Epoch ;
         cgov:epochNumber ?epochNumber .
  OPTIONAL { ?epoch cgov:epochStartTime         ?startTime . }
  OPTIONAL { ?epoch cgov:epochEndTime           ?endTime . }
  OPTIONAL { ?epoch cgov:totalActiveStakeLovelace ?activeStake . }
}
ORDER BY ?epochNumber
"""

run_and_save("Q8", Q8)

Saved 20 rows → /Users/tinayemawocha/Documents/UM/KGS/cardanokb/query_results/Q8.csv


,epoch,epochNumber,startTime,endTime,activeStakeTADA
0,https://w3id.org/cgov#epoch_598,598,1764539091,1764971091,21343
1,https://w3id.org/cgov#epoch_599,599,1764971091,1765403091,21480
2,https://w3id.org/cgov#epoch_600,600,1765403091,1765835091,21398
3,https://w3id.org/cgov#epoch_601,601,1765835091,1766267091,21341
4,https://w3id.org/cgov#epoch_602,602,1766267091,1766699091,21452
5,https://w3id.org/cgov#epoch_603,603,1766699091,1767131091,21475
6,https://w3id.org/cgov#epoch_604,604,1767131091,1767563091,21381
7,https://w3id.org/cgov#epoch_605,605,1767563091,1767995091,21393
8,https://w3id.org/cgov#epoch_606,606,1767995091,1768427091,21385
9,https://w3id.org/cgov#epoch_607,607,1768427091,1768859091,21405
